# PerturBnet Construction — CD4+ T Cell Perturb-seq

**Goal**: Build three independent TF→target gene networks from the Zhu et al. 2025 genome-scale
CD4+ T cell Perturb-seq dataset (Marson lab), one for each culture condition:
- `Rest` — resting CD4+ T cells
- `Stim8hr` — early activation (8-hour stimulation)
- `Stim48hr` — late activation (48-hour stimulation)

**Key design decisions**:
- A TF appears in a given network ONLY if it was perturbed and passed QC **in that condition**.
  No edges are borrowed or shared across conditions.
- No log2FC or p-value threshold is applied — all edges are retained and fully annotated
  so that Mithun can apply cutoffs downstream.
- This network is entirely self-contained within the Perturb-seq dataset.
  It does not interact with the scPRINT or pySCENIC pipelines.

**Output files (written to Drive)**:
- `perturbnet_Rest.csv`
- `perturbnet_Stim8hr.csv`
- `perturbnet_Stim48hr.csv`
- `perturbnet_summary.csv` — per-condition TF coverage summary

**Edge schema** (columns in each output CSV):
- `source_TF` — perturbed transcription factor
- `target_gene` — gene whose expression changed upon TF perturbation
- `log2FC` — log2 fold change of target gene in perturbed vs. control cells
- `adj_p_value` — Benjamini-Hochberg FDR-adjusted p-value (from DESeq2)
- `p_value` — raw p-value
- `zscore` — z-score (log2FC / lfcSE), useful for ranking effect magnitude
- `culture_condition` — condition label (Rest / Stim8hr / Stim48hr)

**Source dataset**: `GWCD4i.DE_stats.h5ad` (Zhu et al. 2025, CZI Virtual Cell Platform)
Accessed via: `s3://genome-scale-tcell-perturb-seq/marson2025_data/`

## 0. Environment Setup

Mount Google Drive and set up cache paths. The DE stats file is large and lives on S3,
so we download it once to a local VM cache directory to avoid re-downloading across cells.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.3 MB/s eta 0:00:00


In [3]:
import boto3

In [4]:
import os

# ── Drive paths (persistent across sessions) ──────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/benchmarking_paper/datasets'
DRIVE_PERTURB = os.path.join(DRIVE_BASE, 'perturbnet_outputs')
TF_LIST_PATH  = os.path.join(DRIVE_BASE, 'humanTFs_1639_clean.csv')  # Lambert et al. 2018

os.makedirs(DRIVE_PERTURB, exist_ok=True)
print(f'Output directory ready: {DRIVE_PERTURB}')

# ── Local VM cache (fast I/O, does not persist after runtime) ─────────────────
CACHE_DIR   = '/content/perturbseq_cache'
DE_CACHE    = os.path.join(CACHE_DIR, 'GWCD4i.DE_stats.h5ad')

os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache directory ready: {CACHE_DIR}')

Output directory ready: /content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs
Cache directory ready: /content/perturbseq_cache


## 1. Download DE Stats File from S3

`GWCD4i.DE_stats.h5ad` contains the differential expression results for every
(perturbed gene × culture condition) pair across all 10,282 measured genes.
This is the only file we need for PerturBnet construction.

Each row in `.obs` = one TF perturbation in one condition.
Each column in `.var` = one measured gene (potential target).
Edge weights live in `.layers['log_fc']`, `.layers['adj_p_value']`, etc.

In [5]:
import os
import sys
import json
import gzip
import shutil
import argparse
import subprocess
import warnings
warnings.filterwarnings("ignore")


parser = argparse.ArgumentParser(description="Compare Perturb-seq vs Quiescence datasets")
parser.add_argument("--skip_download_quiescence", action="store_true",
                    help="Skip downloading the quiescence dataset")
parser.add_argument("--skip_download_perturbseq", action="store_true",
                    help="Skip downloading the Perturb-seq dataset")
parser.add_argument("--download_pseudobulk", action="store_true",
                    help="Also download GWCD4i.pseudobulk_merged.h5ad from S3")
parser.add_argument("--outdir", default="outputs_comparison",
                    help="Output directory (default: outputs_comparison)")
args = parser.parse_args([])

OUTDIR = args.outdir
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs("./my_data_folder", exist_ok=True)
os.makedirs("./perturbseq_data", exist_ok=True)

In [6]:
S3_BUCKET   = "s3://genome-scale-tcell-perturb-seq/marson2025_data"
PS_DE_FILE  = "GWCD4i.DE_stats.h5ad"
PS_PB_FILE  = "GWCD4i.pseudobulk_merged.h5ad"
PS_DE_PATH  = f"./perturbseq_data/{PS_DE_FILE}"
PS_PB_PATH  = f"./perturbseq_data/{PS_PB_FILE}"

def s3_download(s3_uri, local_path, label):
    """Download from S3 using aws CLI (unsigned) or boto3 fallback."""
    if os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / (1024**2)
        print(f"  {label} already exists ({size_mb:.1f} MB) — skipping.")
        return

    # Try aws cli
    aws_ok = shutil.which("aws") is not None
    if aws_ok:
        print(f"  Downloading {label} via aws s3 cp …")
        cmd = ["aws", "s3", "cp", "--no-sign-request", s3_uri, local_path]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            print(f"  ✓ Downloaded: {local_path}")
            return
        else:
            print(f"  aws CLI failed: {result.stderr.strip()}")
            print("  Falling back to boto3…")

    # boto3 fallback
    try:
        import boto3
        from botocore import UNSIGNED
        from botocore.config import Config
    except ImportError:
        print("  [ERROR] boto3 not installed. Install with: pip install boto3")
        sys.exit(1)

    # Parse s3://bucket/key
    s3_path   = s3_uri.replace("s3://", "")
    bucket    = s3_path.split("/")[0]
    key       = "/".join(s3_path.split("/")[1:])

    print(f"  Downloading {label} via boto3 (unsigned)…")
    try:
        s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
        meta = s3.head_object(Bucket=bucket, Key=key)
        total_bytes = meta["ContentLength"]
        total_mb    = total_bytes / (1024**2)
        print(f"  File size: {total_mb:.1f} MB")
        downloaded_bytes = 0

        def progress(bytes_transferred):
            nonlocal downloaded_bytes
            downloaded_bytes += bytes_transferred
            pct = downloaded_bytes / total_bytes * 100
            print(f"  {downloaded_bytes/(1024**2):.1f} / {total_mb:.1f} MB ({pct:.1f}%)", end="\r")

        s3.download_file(bucket, key, local_path, Callback=progress)
        print(f"\n  ✓ Downloaded: {local_path}")
    except Exception as e:
        print(f"  ✗ boto3 download failed: {e}")
        sys.exit(1)

if not args.skip_download_perturbseq:
    print("\n" + "─" * 70)
    print("STEP 1b — Download Perturb-seq dataset from S3")
    print("─" * 70)
    s3_download(f"{S3_BUCKET}/{PS_DE_FILE}", PS_DE_PATH, PS_DE_FILE)
    if args.download_pseudobulk:
        s3_download(f"{S3_BUCKET}/{PS_PB_FILE}", PS_PB_PATH, PS_PB_FILE)
else:
    print("\n[SKIP] Perturb-seq download skipped (--skip_download_perturbseq).")
    if not os.path.exists(PS_DE_PATH):
        print(f"[ERROR] Expected file not found: {PS_DE_PATH}")
        sys.exit(1)


──────────────────────────────────────────────────────────────────────
STEP 1b — Download Perturb-seq dataset from S3
──────────────────────────────────────────────────────────────────────
  File size: 16005.9 MB

  ✓ Downloaded: ./perturbseq_data/GWCD4i.DE_stats.h5ad


In [7]:
# ============================================================
# Build PCA embeddings from GWCD4i.DE_stats.h5ad and save to Drive
# ============================================================

import os
import numpy as np
import pandas as pd

# Install if needed
try:
    import scanpy as sc
except ImportError:
    !pip install -q scanpy anndata
    import scanpy as sc

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Paths
# ----------------------------
PS_DE_PATH = "./perturbseq_data/GWCD4i.DE_stats.h5ad"

OUT_DIR = "/content/drive/MyDrive/benchmarking_paper/datasets/embeddings"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "GWCD4i_DE_stats_Rest_TF_PCA_embeddings.csv")

# ----------------------------
# Load Perturb-seq DE AnnData
# ----------------------------
adata = sc.read_h5ad(PS_DE_PATH)

print("Loaded AnnData:")
print(adata)
print("\nobs columns:")
print(adata.obs.columns.tolist())
print("\nvar columns:")
print(adata.var.columns.tolist())
print("\nlayers:")
print(list(adata.layers.keys()))

# ----------------------------
# Filter to Rest condition
# ----------------------------
if "culture_condition" not in adata.obs.columns:
    raise ValueError("Expected adata.obs['culture_condition'], but it was not found.")

adata_rest = adata[adata.obs["culture_condition"].astype(str) == "Rest"].copy()

print("\nRest subset:")
print(adata_rest)

# ----------------------------
# Identify perturbed TF/gene column
# ----------------------------
possible_tf_cols = [
    "target_contrast_gene_name",
    "target_gene",
    "source_TF",
    "sourceTF",
    "gene",
    "perturbed_gene",
    "target"
]

tf_col = next((c for c in possible_tf_cols if c in adata_rest.obs.columns), None)

if tf_col is None:
    raise ValueError(
        "Could not find perturbed TF/gene column. "
        f"Available obs columns: {adata_rest.obs.columns.tolist()}"
    )

print(f"\nUsing perturbed TF column: {tf_col}")

# ----------------------------
# Choose logFC layer
# ----------------------------
possible_logfc_layers = ["log_fc", "log2FC", "lfc", "logfoldchange"]

logfc_layer = next((l for l in possible_logfc_layers if l in adata_rest.layers.keys()), None)

if logfc_layer is None:
    raise ValueError(
        "Could not find logFC layer. "
        f"Available layers: {list(adata_rest.layers.keys())}"
    )

print(f"Using logFC layer: {logfc_layer}")

# ----------------------------
# Build TF x target-gene matrix
# ----------------------------
X = adata_rest.layers[logfc_layer]

# Convert sparse to dense if needed
if hasattr(X, "toarray"):
    X = X.toarray()

target_genes = adata_rest.var_names.astype(str)
perturbed_tfs = adata_rest.obs[tf_col].astype(str).values

logfc_df = pd.DataFrame(X, index=perturbed_tfs, columns=target_genes)

# If multiple rows per TF exist, average them
logfc_df = logfc_df.groupby(logfc_df.index).mean()

# Clean infinities / NaNs
logfc_df = logfc_df.replace([np.inf, -np.inf], np.nan).fillna(0)

print("\nTF x target-gene logFC matrix:")
print(logfc_df.shape)
display(logfc_df.head())

# ----------------------------
# Standardize and run PCA
# ----------------------------
N_COMPONENTS = min(50, logfc_df.shape[0] - 1, logfc_df.shape[1])

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(logfc_df.values)

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

embedding_cols = [f"PC{i+1}" for i in range(N_COMPONENTS)]

pca_embeddings_df = pd.DataFrame(
    X_pca,
    index=logfc_df.index,
    columns=embedding_cols
)

pca_embeddings_df.index.name = "gene"

# ----------------------------
# Save
# ----------------------------
pca_embeddings_df.to_csv(OUT_CSV)

print("\nSaved PCA embeddings to:")
print(OUT_CSV)

print("\nPCA embedding shape:")
print(pca_embeddings_df.shape)

print("\nExplained variance, first 10 PCs:")
print(pca.explained_variance_ratio_[:10])

display(pca_embeddings_df.head())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Loaded AnnData:
AnnData object with n_obs × n_vars = 33983 × 10282
    obs: 'target_contrast_gene_nam

,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001460,ENSG00000001461,ENSG00000001561,ENSG00000001629,...,ENSG00000280670,ENSG00000280789,ENSG00000281991,ENSG00000284194,ENSG00000284308,ENSG00000284770,ENSG00000285077,ENSG00000288722,ENSG00000290292,ENSG00000291237
A1BG,0.060426,-0.002338,0.087863,0.345917,0.060123,0.028757,-0.052808,-0.132228,-0.064843,0.078255,...,-0.035435,-0.010372,0.199122,-0.011645,0.491165,-0.008909,0.018546,-0.255547,0.007548,-0.012661
A2M,-0.145953,0.151592,-0.140420,2.118994,-0.127941,0.040451,0.275602,0.035576,0.449748,-0.064210,...,-0.652447,-0.135359,0.512875,-0.068182,-1.249371,-0.009719,-0.311281,0.063446,0.047129,-0.095536
AAAS,0.096137,-0.196582,0.144975,-1.400035,0.173641,0.183301,-0.095154,-0.175264,0.052309,-0.030131,...,0.137651,0.051271,0.554700,-0.058526,-0.122295,0.044813,0.092323,-0.003890,-0.130206,0.039722
AACS,0.071413,0.063905,-0.143228,1.036428,-0.069606,0.134166,0.126303,0.071662,0.031638,-0.114042,...,-0.060862,-0.033237,0.090251,-0.103701,1.321024,-0.141703,-0.229005,-0.403973,0.035701,0.077186
AAGAB,0.012245,0.024214,-0.024889,0.188395,-0.039281,0.025766,0.255113,0.043768,-0.033456,-0.099980,...,0.229292,0.042726,-0.106760,0.013682,-0.100221,0.129862,-0.069509,-0.058660,-0.046810,-0.015314



Saved PCA embeddings to:
/content/drive/MyDrive/benchmarking_paper/datasets/embeddings/GWCD4i_DE_stats_Rest_TF_PCA_embeddings.csv

PCA embedding shape:
(11287, 50)

Explained variance, first 10 PCs:
[0.07926485 0.02362548 0.01972794 0.01454259 0.00947158 0.00810181
 0.00691129 0.00593516 0.0056708  0.00480996]


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC41,PC42,PC43,PC44,PC45,PC46,PC47,PC48,PC49,PC50
gene,,,,,,,,,,,,,,,,,,,,,
A1BG,18.086092,-6.848879,7.678924,1.693360,0.283415,2.631166,1.426210,9.821709,-4.595931,-3.698311,...,0.644155,0.190476,-1.149666,0.446706,1.306544,0.469457,0.618067,-2.141016,2.388128,-0.716503
A2M,-50.906020,13.951320,-19.062338,-23.054912,10.729358,-2.165407,1.974355,-5.410113,9.539141,1.866734,...,-1.210254,-1.832093,1.410957,1.934742,1.969512,-4.351799,-3.877204,0.269958,0.884285,1.523285
AAAS,31.239392,-0.624432,-11.583426,9.598887,-1.604052,-1.084696,-17.163971,-4.586830,-0.121790,4.802732,...,-1.363106,3.269127,1.586234,0.096930,-1.812505,-1.564309,-0.671625,0.267062,-0.819615,0.082484
AACS,-10.226383,-11.253404,9.304524,12.946848,2.904920,4.667942,-14.282618,5.298923,2.821474,-3.116329,...,1.181415,0.135713,-3.233011,-0.194434,2.075431,-3.794562,-2.711712,4.192849,0.362588,3.459963
AAGAB,-16.647284,-8.683023,-7.396453,20.234510,-0.096473,16.428826,6.605719,6.408802,-2.323947,-10.555519,...,4.435985,-1.784597,-0.673041,0.942302,-1.932040,-2.056166,0.977055,0.518604,-3.894581,1.262111


In [ ]:
import scanpy as sc

In [ ]:


psdata = sc.read_h5ad("/content/perturbseq_data/GWCD4i.DE_stats.h5ad")
print(f"  ✓ Perturb-seq DE stats: {psdata.n_obs:,} rows × {psdata.n_vars:,} genes")
print(f"    (rows = perturbation-condition pairs; columns = genes)")

  ✓ Perturb-seq DE stats: 33,983 rows × 10,282 genes
    (rows = perturbation-condition pairs; columns = genes)


## 2. Load DE Stats AnnData

We load the full DE stats object. Key structure:
- `psdata.obs` — one row per (perturbed_gene × culture_condition), n=33,983
- `psdata.var` — one row per measured gene, n=10,282
- `psdata.layers['log_fc']` — log2FC matrix, shape (n_perturbations × n_genes)
- `psdata.layers['adj_p_value']` — FDR-adjusted p-values (BH), same shape
- `psdata.layers['p_value']` — raw p-values
- `psdata.layers['zscore']` — z-scores (log2FC / lfcSE)

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
from scipy.sparse import issparse

print('Loading DE stats AnnData...')
print(f'Loaded: {psdata.shape[0]:,} perturbation-condition rows × {psdata.shape[1]:,} genes')
print(f'\nObs columns:\n{list(psdata.obs.columns)}')
print(f'\nAvailable layers: {list(psdata.layers.keys())}')
print(f'\nCulture conditions present: {psdata.obs["culture_condition"].value_counts().to_dict()}')

Loading DE stats AnnData...
Loaded: 33,983 perturbation-condition rows × 10,282 genes

Obs columns:
['target_contrast_gene_name', 'culture_condition', 'target_contrast', 'chunk', 'n_cells_target', 'n_up_genes', 'n_down_genes', 'n_total_de_genes', 'ontarget_effect_size', 'ontarget_significant', 'target_baseMean', 'offtarget_flag', 'n_total_genes_category', 'ontarget_effect_category', 'n_downstream']

Available layers: ['adj_p_value', 'baseMean', 'lfcSE', 'log_fc', 'p_value', 'zscore']

Culture conditions present: {'Stim8hr': 11415, 'Rest': 11287, 'Stim48hr': 11281}


## 3. Load Lambert TF List

We use the Lambert et al. 2018 curated list of 1,639 human TFs (`humanTFs_1639_clean.csv`)
to determine which perturbed genes in the Perturb-seq data are bona fide transcription factors.
Only TFs in this list will be included as source nodes in PerturBnet.

In [ ]:
tf_df = pd.read_csv(TF_LIST_PATH)

# The gene symbol column should be 'gene_symbol' — confirm and extract
print(f'TF list columns: {list(tf_df.columns)}')
tf_set = set(tf_df['gene_symbol'].dropna().str.strip())
print(f'Total human TFs in Lambert list: {len(tf_set)}')

TF list columns: ['gene_id', 'gene_symbol']
Total human TFs in Lambert list: 1639


## 4. QC Filtering

For each perturbation entry in `.obs`, we apply two QC filters from the dataset:

- `ontarget_significant == True`: The guide RNA actually knocked down its intended target gene
  significantly (10% FDR on the on-target effect). This ensures we're seeing real perturbation effects.

- `offtarget_flag == False`: The guide RNA did not show evidence of off-target effects
  (no TSS within 10 kb with significant downregulation of a nearby non-target gene).
  This protects against confounded edge calls.

These are the same filters used in the existing perturb-net notebook.

In [ ]:
obs = psdata.obs.copy()
print(f'Total perturbation-condition entries: {len(obs):,}')

# Apply QC filters
qc_mask = (obs['ontarget_significant'] == True) & (obs['offtarget_flag'] == False)
obs_qc = obs[qc_mask].copy()
print(f'After QC (ontarget_significant & ~offtarget_flag): {len(obs_qc):,} entries')
print(f'Removed: {len(obs) - len(obs_qc):,} entries')

# Show per-condition breakdown after QC
print(f'\nPer-condition counts after QC:')
print(obs_qc['culture_condition'].value_counts())

Total perturbation-condition entries: 33,983
After QC (ontarget_significant & ~offtarget_flag): 18,763 entries
Removed: 15,220 entries

Per-condition counts after QC:
culture_condition
Stim48hr    6410
Stim8hr     6290
Rest        6063
Name: count, dtype: int64


## 5. TF Intersection per Condition

Now we identify which perturbed genes in each condition are TFs according to the Lambert list.
A TF is included in a condition's network ONLY if:
1. It passed QC in that specific condition (not just any condition)
2. Its gene symbol appears in the Lambert et al. TF list

This enforces strict condition-specificity — no borrowing of TFs across Rest/Stim8hr/Stim48hr.

In [ ]:
CONDITIONS = ['Rest', 'Stim8hr', 'Stim48hr']

condition_tf_map = {}  # condition -> list of TF names that passed QC in that condition

for cond in CONDITIONS:
    # Rows in obs that belong to this condition and passed QC
    cond_obs = obs_qc[obs_qc['culture_condition'] == cond]

    # Perturbed gene names in this condition
    perturbed_genes = set(cond_obs['target_contrast_gene_name'].dropna())

    # Intersect with Lambert TF list
    tfs_in_cond = sorted(perturbed_genes & tf_set)
    condition_tf_map[cond] = tfs_in_cond

    print(f'{cond}: {len(perturbed_genes):,} QC-passed perturbations → {len(tfs_in_cond)} are TFs')

# TFs present in all 3 conditions vs. condition-specific
all_three = set(condition_tf_map['Rest']) & set(condition_tf_map['Stim8hr']) & set(condition_tf_map['Stim48hr'])
print(f'\nTFs present in all 3 conditions: {len(all_three)}')
print(f'Rest-only TFs: {len(set(condition_tf_map["Rest"]) - set(condition_tf_map["Stim8hr"]) - set(condition_tf_map["Stim48hr"]))}')

Rest: 6,063 QC-passed perturbations → 561 are TFs
Stim8hr: 6,290 QC-passed perturbations → 596 are TFs
Stim48hr: 6,410 QC-passed perturbations → 576 are TFs

TFs present in all 3 conditions: 481
Rest-only TFs: 20


## 6. Extract Target Gene Measurements

Gene names for the columns (target genes) come from `psdata.var['gene_name']`.
We pre-extract the layer matrices once here to avoid repeated indexing inside the loop.

Each layer is shape (n_perturbations, n_genes). We convert from sparse to dense
in controlled chunks per condition to manage memory.

In [ ]:
# Target gene names (columns of the layer matrices)
target_gene_names = psdata.var['gene_name'].values
print(f'Number of measurable target genes: {len(target_gene_names):,}')

# Index from perturbed gene name -> row indices in psdata (for fast lookup)
# We build a dict: gene_name -> list of integer row positions
obs_full = psdata.obs.copy()  # full obs (before QC) for row indexing
obs_full['_row_idx'] = range(len(obs_full))

gene_to_rows = obs_full.groupby(['target_contrast_gene_name', 'culture_condition'])['_row_idx'].apply(list).to_dict()
print(f'Built row index for {len(gene_to_rows):,} (gene, condition) pairs')

Number of measurable target genes: 10,282
Built row index for 34,578 (gene, condition) pairs


## 7. Build PerturBnet — One Network per Condition

For each condition:
1. Iterate over every QC-passing TF in that condition
2. Find its rows in the full psdata matrix (there may be multiple guides per gene — we average)
3. Extract log2FC, adj_p_value, p_value, zscore for all target genes
4. Build a long-format edge table: one row per (TF, target_gene) pair
5. Remove self-edges (TF→TF where source == target)
6. Annotate with condition label
7. Save immediately to Drive

**No threshold is applied** — all edges are kept regardless of log2FC or p-value magnitude.
Edge annotations allow downstream filtering at any stringency.

In [ ]:
import gc

summary_rows = []  # collect per-condition summary stats

for cond in CONDITIONS:
    print(f'\n{'='*60}')
    print(f'Building PerturBnet for condition: {cond}')
    print(f'{'='*60}')

    tfs = condition_tf_map[cond]
    print(f'TFs to process: {len(tfs)}')

    edge_records = []

    for i, tf in enumerate(tfs):
        if i % 50 == 0:
            print(f'  Processing TF {i+1}/{len(tfs)}: {tf}')

        # Get row indices for this TF in this condition
        row_key = (tf, cond)
        if row_key not in gene_to_rows:
            # This shouldn't happen given our obs_qc filtering, but guard anyway
            print(f'  WARNING: no rows found for {tf} in {cond} — skipping')
            continue

        row_idxs = gene_to_rows[row_key]

        # Additionally verify these rows passed QC
        # (gene_to_rows was built from full obs, so we need to check)
        qc_row_idxs = [
            idx for idx in row_idxs
            if obs_full.iloc[idx]['ontarget_significant'] == True
            and obs_full.iloc[idx]['offtarget_flag'] == False
        ]

        if len(qc_row_idxs) == 0:
            continue

        # Extract layer values for these rows — convert sparse slice to dense
        def extract_layer(layer_name, idxs):
            mat = psdata.layers[layer_name][idxs, :]
            if issparse(mat):
                mat = mat.toarray()
            return mat  # shape: (n_rows, n_genes)

        logfc_mat   = extract_layer('log_fc',      qc_row_idxs)  # (n_rows, n_genes)
        adjpval_mat = extract_layer('adj_p_value', qc_row_idxs)
        pval_mat    = extract_layer('p_value',     qc_row_idxs)
        zscore_mat  = extract_layer('zscore',      qc_row_idxs)

        # If multiple guide rows for the same TF, average across them
        # (this is standard practice — multiple guides reduce noise)
        logfc_vec   = logfc_mat.mean(axis=0)    # (n_genes,)
        adjpval_vec = adjpval_mat.mean(axis=0)
        pval_vec    = pval_mat.mean(axis=0)
        zscore_vec  = zscore_mat.mean(axis=0)

        # Build edge records for this TF — one per target gene
        for j, target in enumerate(target_gene_names):
            # Remove self-edges
            if target == tf:
                continue

            edge_records.append({
                'source_TF':        tf,
                'target_gene':      target,
                'log2FC':           logfc_vec[j],
                'adj_p_value':      adjpval_vec[j],
                'p_value':          pval_vec[j],
                'zscore':           zscore_vec[j],
                'culture_condition': cond,
                'n_guides_averaged': len(qc_row_idxs),
            })

    # Build DataFrame
    net_df = pd.DataFrame(edge_records)
    print(f'\nCondition {cond}: {len(net_df):,} total edges across {net_df["source_TF"].nunique()} TFs')

    # Save to Drive immediately
    out_path = os.path.join(DRIVE_PERTURB, f'perturbnet_{cond}.csv')
    net_df.to_csv(out_path, index=False)
    print(f'Saved to Drive: {out_path}')

    # Collect summary stats
    summary_rows.append({
        'condition':     cond,
        'n_TFs':         net_df['source_TF'].nunique(),
        'n_target_genes': net_df['target_gene'].nunique(),
        'n_edges':       len(net_df),
        'tf_list':       ', '.join(sorted(net_df['source_TF'].unique())),
    })

    # Free memory before next condition
    del net_df, edge_records
    gc.collect()

print('\nAll three PerturBnet networks saved to Drive.')


Building PerturBnet for condition: Rest
TFs to process: 561
  Processing TF 1/561: ADNP
  Processing TF 51/561: DMTF1
  Processing TF 101/561: HOXB4
  Processing TF 151/561: MXD4
  Processing TF 201/561: PURA
  Processing TF 251/561: STAT6
  Processing TF 301/561: ZBTB38
  Processing TF 351/561: ZNF17
  Processing TF 401/561: ZNF316
  Processing TF 451/561: ZNF519
  Processing TF 501/561: ZNF671
  Processing TF 551/561: ZNF92

Condition Rest: 5,767,641 total edges across 561 TFs
Saved to Drive: /content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs/perturbnet_Rest.csv

Building PerturBnet for condition: Stim8hr
TFs to process: 596
  Processing TF 1/596: ADNP
  Processing TF 51/596: DEAF1
  Processing TF 101/596: GTF2I
  Processing TF 151/596: MBD1
  Processing TF 201/596: PCGF6
  Processing TF 251/596: SMAD4
  Processing TF 301/596: USF3
  Processing TF 351/596: ZHX2
  Processing TF 401/596: ZNF26
  Processing TF 451/596: ZNF414
  Processing TF 501/596: ZNF583
  Process

## 8. Save Summary Table

A per-condition summary showing how many TFs, target genes, and total edges
are in each network. Also records which TFs are in each condition for quick reference.

In [ ]:
summary_df = pd.DataFrame(summary_rows)
print(summary_df[['condition', 'n_TFs', 'n_target_genes', 'n_edges']])

summary_path = os.path.join(DRIVE_PERTURB, 'perturbnet_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'\nSummary saved to: {summary_path}')

  condition  n_TFs  n_target_genes  n_edges
0      Rest    561           10282  5767641
1   Stim8hr    596           10282  6127476
2  Stim48hr    576           10282  5921856

Summary saved to: /content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs/perturbnet_summary.csv


## 9. Quick Sanity Checks

Verify the outputs look reasonable before closing the session:
- Check a few canonical T cell activation TFs are present in the right conditions
- Confirm no self-edges survived
- Preview the top edges by absolute log2FC for each condition

In [ ]:
# Canonical T cell TFs we expect to see perturbed in activation conditions
# (from pySCENIC deltaRSS top hits and literature)
CANONICAL_ACT_TFS = ['FOSB', 'BATF', 'BATF3', 'IRF4', 'RELB', 'ETV6', 'E2F4']
CANONICAL_REST_TFS = ['KLF2', 'TCF7', 'MYBL1', 'FOXO1', 'LEF1']

print('=== TF presence check across conditions ===')
for cond in CONDITIONS:
    net = pd.read_csv(os.path.join(DRIVE_PERTURB, f'perturbnet_{cond}.csv'))
    tfs_in_net = set(net['source_TF'].unique())

    print(f'\n[{cond}]')
    print(f'  Total TFs: {len(tfs_in_net)}')

    # Check canonical activation TFs
    act_present = [tf for tf in CANONICAL_ACT_TFS if tf in tfs_in_net]
    print(f'  Canonical activation TFs present: {act_present}')

    # Check canonical resting TFs
    rest_present = [tf for tf in CANONICAL_REST_TFS if tf in tfs_in_net]
    print(f'  Canonical resting TFs present: {rest_present}')

    # Confirm no self-edges
    self_edges = net[net['source_TF'] == net['target_gene']]
    print(f'  Self-edges (should be 0): {len(self_edges)}')

    # Top 5 edges by absolute log2FC for this condition
    print(f'  Top 5 edges by |log2FC|:')
    top5 = net.reindex(net['log2FC'].abs().nlargest(5).index)[['source_TF','target_gene','log2FC','adj_p_value']]
    print(top5.to_string(index=False))

    del net
    gc.collect()

=== TF presence check across conditions ===

[Rest]
  Total TFs: 561
  Canonical activation TFs present: ['BATF', 'BATF3', 'RELB', 'ETV6', 'E2F4']
  Canonical resting TFs present: ['KLF2', 'TCF7', 'MYBL1', 'FOXO1']
  Self-edges (should be 0): 0
  Top 5 edges by |log2FC|:
source_TF target_gene    log2FC  adj_p_value
     ARNT         LYZ -9.415596     0.000039
     KAT7       CRTAM  8.818204     0.000004
    AEBP2        TRDC -7.731936     0.740052
    NAIF1        MT1H  7.564668     0.001893
    NCOA2        TRDC -7.110236     0.999961

[Stim8hr]
  Total TFs: 596
  Canonical activation TFs present: ['BATF', 'BATF3', 'IRF4', 'RELB', 'ETV6', 'E2F4']
  Canonical resting TFs present: ['TCF7', 'FOXO1']
  Self-edges (should be 0): 0
  Top 5 edges by |log2FC|:
source_TF target_gene     log2FC  adj_p_value
    RBCK1        IL22 -14.523402 9.999821e-01
   ZNF653       IL17F  14.404570 1.450405e-11
     USF3      IL17RB  12.698030 9.191631e-10
     MTF1         IL4  -9.275656 9.157144e-01
   ZBT

## 10. TF Coverage Summary — Cross-Condition

Which TFs appear in all three conditions? Which are condition-specific?
This informs which TFs could be compared across conditions later when building T-net.

In [ ]:
tf_sets = {}
for cond in CONDITIONS:
    net = pd.read_csv(os.path.join(DRIVE_PERTURB, f'perturbnet_{cond}.csv'))
    tf_sets[cond] = set(net['source_TF'].unique())
    del net

in_all_three       = tf_sets['Rest'] & tf_sets['Stim8hr'] & tf_sets['Stim48hr']
rest_only          = tf_sets['Rest'] - tf_sets['Stim8hr'] - tf_sets['Stim48hr']
stim8_only         = tf_sets['Stim8hr'] - tf_sets['Rest'] - tf_sets['Stim48hr']
stim48_only        = tf_sets['Stim48hr'] - tf_sets['Rest'] - tf_sets['Stim8hr']
act_both_not_rest  = tf_sets['Stim8hr'] & tf_sets['Stim48hr'] - tf_sets['Rest']

print(f'TFs present in ALL 3 conditions:      {len(in_all_three)}')
print(f'TFs present in Rest only:             {len(rest_only)}')
print(f'TFs present in Stim8hr only:          {len(stim8_only)}')
print(f'TFs present in Stim48hr only:         {len(stim48_only)}')
print(f'TFs in both activation, not rest:     {len(act_both_not_rest)}')

print(f'\nTFs in all 3 conditions (these are the most robust candidates):')
print(sorted(in_all_three))

TFs present in ALL 3 conditions:      481
TFs present in Rest only:             20
TFs present in Stim8hr only:          37
TFs present in Stim48hr only:         33
TFs in both activation, not rest:     40

TFs in all 3 conditions (these are the most robust candidates):
['ADNP', 'ADNP2', 'AEBP2', 'AHDC1', 'AHR', 'AKAP8L', 'ARID2', 'ARID3A', 'ARID3B', 'ARID5A', 'ARNT', 'ASH1L', 'ATF1', 'ATF2', 'ATF4', 'ATF6', 'ATF6B', 'ATF7', 'ATMIN', 'BATF', 'BATF3', 'BAZ2A', 'BBX', 'BCL6', 'BHLHE40', 'BPTF', 'BRF2', 'CAMTA1', 'CC2D1A', 'CEBPB', 'CENPB', 'CENPS', 'CHAMP1', 'CIC', 'CLOCK', 'CREB1', 'CREBL2', 'CREBZF', 'CREM', 'CSRNP1', 'CSRNP2', 'CUX1', 'CXXC1', 'DEAF1', 'DMTF1', 'DNTTIP1', 'DOT1L', 'DR1', 'E2F1', 'E2F3', 'E2F4', 'E2F6', 'E2F7', 'E2F8', 'EEA1', 'ELF4', 'ELK3', 'ELK4', 'ERF', 'ESRRA', 'ETS2', 'ETV3', 'FAM200B', 'FBXL19', 'FLI1', 'FOSL2', 'FOXJ2', 'FOXJ3', 'FOXK1', 'FOXK2', 'FOXN2', 'FOXO1', 'FOXO4', 'FOXP3', 'FOXP4', 'GABPA', 'GATA3', 'GATAD2A', 'GATAD2B', 'GLI4', 'GLMP', 'GMEB1', 'GPBP1

In [ ]:
# ── Re-run Stim48hr only — save to local Colab first ─────────────────────────
import gc
from scipy.sparse import issparse

cond = 'Stim48hr'
tfs = condition_tf_map[cond]
print(f'Processing {len(tfs)} TFs for {cond}...')

edge_records = []

for i, tf in enumerate(tfs):
    if i % 50 == 0:
        print(f'  {i+1}/{len(tfs)}: {tf}')

    row_key = (tf, cond)
    if row_key not in gene_to_rows:
        continue

    qc_row_idxs = [
        idx for idx in gene_to_rows[row_key]
        if obs_full.iloc[idx]['ontarget_significant'] == True
        and obs_full.iloc[idx]['offtarget_flag'] == False
    ]
    if len(qc_row_idxs) == 0:
        continue

    def extract_layer(layer_name, idxs):
        mat = psdata.layers[layer_name][idxs, :]
        if issparse(mat): mat = mat.toarray()
        return mat

    logfc_vec   = extract_layer('log_fc',      qc_row_idxs).mean(axis=0)
    adjpval_vec = extract_layer('adj_p_value', qc_row_idxs).mean(axis=0)
    pval_vec    = extract_layer('p_value',     qc_row_idxs).mean(axis=0)
    zscore_vec  = extract_layer('zscore',      qc_row_idxs).mean(axis=0)

    for j, target in enumerate(target_gene_names):
        if target == tf:
            continue
        edge_records.append({
            'source_TF':         tf,
            'target_gene':       target,
            'log2FC':            logfc_vec[j],
            'adj_p_value':       adjpval_vec[j],
            'p_value':           pval_vec[j],
            'zscore':            zscore_vec[j],
            'culture_condition': cond,
            'n_guides_averaged': len(qc_row_idxs),
        })

net_df = pd.DataFrame(edge_records)
print(f'{len(net_df):,} edges across {net_df["source_TF"].nunique()} TFs')

# ── Step 1: Save to local Colab VM ────────────────────────────────────────────
LOCAL_PATH = f'/content/perturbnet_{cond}.csv'
net_df.to_csv(LOCAL_PATH, index=False)
local_size = os.path.getsize(LOCAL_PATH) / 1e6
print(f'Saved locally: {LOCAL_PATH} ({local_size:.1f} MB)')

# ── Step 2: Only copy to Drive once local save is confirmed ───────────────────
# Free up Drive space first, then run this block
# DRIVE_PATH = os.path.join(DRIVE_PERTURB, f'perturbnet_{cond}.csv')
# import shutil
# shutil.copy(LOCAL_PATH, DRIVE_PATH)
# print(f'Copied to Drive: {DRIVE_PATH}')

Processing 576 TFs for Stim48hr...
  1/576: ADNP
  51/576: DNTTIP1
  101/576: HBP1
  151/576: MSANTD4
  201/576: PRMT3
  251/576: STAT2
  301/576: ZBTB33
  351/576: ZNF180
  401/576: ZNF318
  451/576: ZNF510
  501/576: ZNF641
  551/576: ZNF790
5,921,856 edges across 576 TFs
Saved locally: /content/perturbnet_Stim48hr.csv (588.5 MB)
